# COMPUTER VISION - LAB 02
## Image Processing Fundamentals
**Week 2 | Python + OpenCV**  
**CLO Alignment:** CLO1, CLO2  

---

### 1. Lab Description
In this lab, students implement fundamental image processing operations using Python and OpenCV. The practical work focuses on smoothing, Gaussian filtering, sharpening, histogram enhancement, thresholding, and morphological operations. Students then apply different filters to sample images and compare their effects.

### 2. Learning Objectives
- Explain the purpose of basic image processing operations.
- Apply smoothing filters using OpenCV.
- Apply Gaussian filtering using OpenCV.
- Apply sharpening techniques to enhance image details.
- Perform histogram enhancement.
- Apply thresholding to create a binary image.
- Apply basic morphological operations.
- Compare the effects of different image processing techniques.
- Interpret how each operation changes visual information.

### 3. CLO Alignment
| CLO | Relation to This Lab |
| :--- | :--- |
| **CLO1** | Explain principles, mathematical foundations, techniques, and applications of digital image processing. |
| **CLO2** | Apply image processing/CV techniques for visual analysis. |

### 4. Recommended Folder Structure
```text
Lab_02_Image_Processing/
├── images/
│   └── sample.jpg
├── outputs/
└── lab02.ipynb
```

In [ ]:
# Required Software and Libraries installation
# !pip install opencv-python numpy matplotlib

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print("OpenCV version:", cv2.__version__)
print("NumPy version:", np.__version__)

### 6. Background Theory

#### 6.1 What Is Image Processing?
Image processing is the application of computational and mathematical operations to a digital image to modify, enhance, transform, or analyze its pixel values.

$$\text{Input Image} \longrightarrow \text{Image Processing Operation} \longrightarrow \text{Processed Image}$$

#### 6.2 Main Operations in This Lab
| Operation | Main Purpose |
| :--- | :--- |
| **Smoothing** | Reduce local variations and small unwanted fluctuations. |
| **Gaussian filtering** | Perform weighted smoothing. |
| **Sharpening** | Emphasize edges and fine details. |
| **Histogram enhancement** | Improve intensity distribution/contrast. |
| **Thresholding** | Separate regions based on intensity. |
| **Morphological operations** | Modify/analyze shapes and structures, especially in binary images. |

#### 6.3 Filtering and Neighborhoods
Many image filters operate on a pixel and its neighboring pixels. A $3 \times 3$ neighborhood contains nine pixels. A mathematical rule is applied to the neighborhood to produce an output value.

$$\text{Image Neighborhood} \longrightarrow \text{Filter} \longrightarrow \text{New Pixel Value}$$

--- 
## Part A — Initial Setup

In [ ]:
# Create directories if they do not exist
os.makedirs("images", exist_ok=True)
os.makedirs("outputs", exist_ok=True)

# Fallback synthetic image generator if sample.jpg is not present
if not os.path.exists("images/sample.jpg"):
    sample_canvas = np.zeros((300, 300, 3), dtype=np.uint8)
    sample_canvas[:] = (200, 200, 200)
    cv2.circle(sample_canvas, (150, 150), 80, (50, 50, 200), -1)
    cv2.putText(sample_canvas, 'Sample', (75, 160), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.imwrite("images/sample.jpg", sample_canvas)
    print("Default images/sample.jpg created.")

# Load the sample image
image = cv2.imread("images/sample.jpg")

if image is None:
    print("Error: Image could not be loaded.")
else:
    print("Image loaded successfully.")

# Convert BGR to RGB for correct Matplotlib display
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 5))
plt.imshow(image_rgb)
plt.title("Original Image")
plt.axis("off")
plt.show()

> **Expected Result:** The original image should appear normally in color. OpenCV reads color images in BGR order, while Matplotlib expects RGB for standard color display.

--- 
## Part B — Image Smoothing

### B1. What Is Smoothing?
Smoothing reduces small local variations in an image. It can reduce noise and produce a softer image, but it may also weaken edges and remove fine details.

$$\text{Original Image} \longrightarrow \text{Smoothing Filter} \longrightarrow \text{Reduced Local Variation} \longrightarrow \text{Smoother Image}$$

### B2. Mean / Average Filtering
A mean filter replaces a pixel with the average of neighboring pixels. A common $3 \times 3$ kernel is:
$$K = \frac{1}{9}\begin{bmatrix} 1 & 1 & 1 \\ 1 & 1 & 1 \\ 1 & 1 & 1 \end{bmatrix}$$

**Daily-life example:** Averaging several nearby measurements can reduce small fluctuations.

In [ ]:
mean_filtered = cv2.blur(image, (5, 5))
mean_rgb = cv2.cvtColor(mean_filtered, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 5))
plt.imshow(mean_rgb)
plt.title("Mean Filter (5x5)")
plt.axis("off")
plt.show()

**Question:** What happens to fine details when a mean filter is applied?  
*Answer:* 

--- 
## Part C — Gaussian Filtering

### C1. Gaussian Filtering
Gaussian filtering is a smoothing technique in which neighboring pixels receive different weights. Pixels closer to the center generally have greater influence.

A simplified $3 \times 3$ Gaussian kernel is:
$$K = \frac{1}{16}\begin{bmatrix} 1 & 2 & 1 \\ 2 & 4 & 2 \\ 1 & 2 & 1 \end{bmatrix}$$

In [ ]:
gaussian_filtered = cv2.GaussianBlur(image, (5, 5), 0)
gaussian_rgb = cv2.cvtColor(gaussian_filtered, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 5))
plt.imshow(gaussian_rgb)
plt.title("Gaussian Filter (5x5)")
plt.axis("off")
plt.show()

### C2. Compare Mean and Gaussian Filtering

In [ ]:
plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(image_rgb)
plt.title("Original")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(mean_rgb)
plt.title("Mean Filter")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(gaussian_rgb)
plt.title("Gaussian Filter")
plt.axis("off")

plt.tight_layout()
plt.show()

| Image | Observation |
| :--- | :--- |
| **Original** | Baseline reference with sharp features and full detail. |
| **Mean Filter** | Uniformly blurred; high-frequency edge information is softened indiscriminately. |
| **Gaussian Filter** | Weighted blur; edge transitions appear smoother and more natural than uniform averaging. |

- **Which image appears smoother?**  
- **What happened to fine details?**  
- **What happened to edges?**  
- **How does Gaussian filtering differ from simple averaging?** 

--- 
## Part D — Image Sharpening

### D1. What Is Sharpening?
Sharpening enhances the appearance of edges and fine details by emphasizing intensity changes. Aggressive sharpening can also make noise more visible.

**Daily-life example:** Sharpening a slightly soft photograph of printed text can make letter boundaries more distinct.

### D2. Sharpening Using a Kernel
A commonly used sharpening kernel is:
$$K = \begin{bmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0 \end{bmatrix}$$

In [ ]:
sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

sharpened = cv2.filter2D(image, -1, sharpen_kernel)
sharpened_rgb = cv2.cvtColor(sharpened, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(6, 5))
plt.imshow(sharpened_rgb)
plt.title("Sharpened Image")
plt.axis("off")
plt.show()

> **Observe:** Object boundaries, hair/fur, text, fine textures, and background noise.

--- 
## Part E — Histogram Enhancement

### E1. What Is an Image Histogram?
An image histogram describes the distribution of intensity values in an image. For an 8-bit grayscale image, possible intensity values range from 0 to 255 ($0 = \text{black}, 255 = \text{white}$).

### E2. Histogram Equalization
Histogram equalization is a common enhancement technique that redistributes intensity values across the full dynamic range to improve contrast.

In [ ]:
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
equalized = cv2.equalizeHist(gray)

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.imshow(gray, cmap="gray")
plt.title("Original Grayscale")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(equalized, cmap="gray")
plt.title("Histogram Equalized")
plt.axis("off")

plt.tight_layout()
plt.show()

### E3. Display the Histograms

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.hist(gray.ravel(), 256, [0, 256], color='steelblue')
plt.title("Original Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
plt.hist(equalized.ravel(), 256, [0, 256], color='darkorange')
plt.title("Equalized Histogram")
plt.xlabel("Pixel Intensity")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

- **Did the image contrast change?**  
- **How did the histogram distribution change?**  
- **Is histogram equalization always visually better?** 

--- 
## Part F — Thresholding

### F1. What Is Thresholding?
Thresholding separates pixels according to intensity. For threshold $T$:
$$I(x,y) \ge T \implies 255, \quad I(x,y) < T \implies 0$$

$$\text{Grayscale Image} \longrightarrow \text{Threshold } T \longrightarrow \text{Binary Image}$$

### F2. Simple Thresholding

In [ ]:
ret, binary = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(6, 5))
plt.imshow(binary, cmap="gray")
plt.title("Binary Threshold (T=127)")
plt.axis("off")
plt.show()

### F3. Experiment With Different Threshold Values

In [ ]:
_, binary_80 = cv2.threshold(gray, 80, 255, cv2.THRESH_BINARY)
_, binary_150 = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY)

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(gray, cmap="gray")
plt.title("Grayscale")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(binary_80, cmap="gray")
plt.title("Threshold = 80")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(binary_150, cmap="gray")
plt.title("Threshold = 150")
plt.axis("off")

plt.tight_layout()
plt.show()

> **Observation:** Changing the threshold changes which pixels are classified as foreground and background.

--- 
## Part G — Morphological Operations

### G1. What Is Mathematical Morphology?
Morphological operations modify or analyze the shape and structure of objects, particularly in binary images. They use a small pattern called a structuring element.

| Operation | Sequence | Typical Effect |
| :--- | :--- | :--- |
| **Erosion** | — | Shrink foreground |
| **Dilation** | — | Expand foreground |
| **Opening** | Erosion $\rightarrow$ Dilation | Remove small isolated foreground structures |
| **Closing** | Dilation $\rightarrow$ Erosion | Close small gaps/holes |

In [ ]:
kernel = np.ones((5, 5), np.uint8)

# G2. Erosion
eroded = cv2.erode(binary, kernel, iterations=1)

# G3. Dilation
dilated = cv2.dilate(binary, kernel, iterations=1)

# G4. Opening
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

# G5. Closing
closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes[0, 0].imshow(eroded, cmap="gray")
axes[0, 0].set_title("Erosion")
axes[0, 0].axis("off")

axes[0, 1].imshow(dilated, cmap="gray")
axes[0, 1].set_title("Dilation")
axes[0, 1].axis("off")

axes[1, 0].imshow(opening, cmap="gray")
axes[1, 0].set_title("Opening (Erode -> Dilate)")
axes[1, 0].axis("off")

axes[1, 1].imshow(closing, cmap="gray")
axes[1, 1].set_title("Closing (Dilate -> Erode)")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

--- 
## Part H — Compare All Processing Results

In [ ]:
plt.figure(figsize=(15, 12))

plt.subplot(3, 3, 1); plt.imshow(image_rgb); plt.title("Original"); plt.axis("off")
plt.subplot(3, 3, 2); plt.imshow(mean_rgb); plt.title("Mean"); plt.axis("off")
plt.subplot(3, 3, 3); plt.imshow(gaussian_rgb); plt.title("Gaussian"); plt.axis("off")
plt.subplot(3, 3, 4); plt.imshow(sharpened_rgb); plt.title("Sharpened"); plt.axis("off")
plt.subplot(3, 3, 5); plt.imshow(equalized, cmap="gray"); plt.title("Histogram Equalized"); plt.axis("off")
plt.subplot(3, 3, 6); plt.imshow(binary, cmap="gray"); plt.title("Threshold"); plt.axis("off")
plt.subplot(3, 3, 7); plt.imshow(eroded, cmap="gray"); plt.title("Erosion"); plt.axis("off")
plt.subplot(3, 3, 8); plt.imshow(dilated, cmap="gray"); plt.title("Dilation"); plt.axis("off")

plt.tight_layout()
plt.show()

### Comparison Table

| Operation | Main Purpose | What Changed? | Effect on Edges | Effect on Details |
| :--- | :--- | :--- | :--- | :--- |
| **Mean** | Unweighted local smoothing | Intensity variations smoothed out uniformly | Blurred / softened | Fine details attenuated |
| **Gaussian** | Distance-weighted smoothing | Noise reduced with higher central weighting | Smoothed naturally | High frequency details reduced |
| **Sharpening** | Edge enhancement | High-frequency gradient amplified | Sharpened / more prominent | Details enhanced, noise elevated |
| **Histogram Enhancement** | Dynamic range stretch | Contrast enhanced across full 0-255 spectrum | Contrast increased | Hidden shadow/highlight details exposed |
| **Thresholding** | Segmentation into binary classes | Pixel intensities quantized to 0 or 255 | Replaced with strict binary boundaries | Grayscale gradient details discarded |
| **Erosion** | Foreground shrinkage | Foreground boundaries eroded inward | Boundary shrunk | Small foreground artifacts eliminated |
| **Dilation** | Foreground expansion | Foreground boundaries dilated outward | Boundary expanded | Small holes filled, regions merged |

--- 
## Part I — Independent Practical Task

Apply all operations to an independent sample image (`images/sample2.jpg`).

In [ ]:
# Create/load sample2.jpg
task_img_path = "images/sample2.jpg"
if not os.path.exists(task_img_path):
    # Synthetic fallback
    sample2 = np.zeros((300, 300, 3), dtype=np.uint8)
    sample2[50:250, 50:250] = (180, 120, 70)
    cv2.rectangle(sample2, (90, 90), (210, 210), (220, 220, 220), -1)
    cv2.imwrite(task_img_path, sample2)

img2 = cv2.imread(task_img_path)
img2_rgb = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

# 1. 5x5 Mean filter
img2_mean = cv2.blur(img2, (5, 5))

# 2. 5x5 Gaussian filter
img2_gauss = cv2.GaussianBlur(img2, (5, 5), 0)

# 3. Sharpening
k_sharp = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
img2_sharp = cv2.filter2D(img2, -1, k_sharp)

# 4. Grayscale
img2_gray = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)

# 5. Histogram Equalization
img2_eq = cv2.equalizeHist(img2_gray)

# 6. Thresholding
_, img2_thresh = cv2.threshold(img2_gray, 127, 255, cv2.THRESH_BINARY)

# 7. Erosion & 8. Dilation
k_morph = np.ones((5, 5), np.uint8)
img2_erode = cv2.erode(img2_thresh, k_morph, iterations=1)
img2_dilate = cv2.dilate(img2_thresh, k_morph, iterations=1)

# 9. Display All Results
titles = ["Original", "Mean 5x5", "Gaussian 5x5", "Sharpened", 
          "Grayscale", "Equalized", "Threshold (127)", "Erosion", "Dilation"]
images_list = [img2_rgb, cv2.cvtColor(img2_mean, cv2.COLOR_BGR2RGB), 
               cv2.cvtColor(img2_gauss, cv2.COLOR_BGR2RGB), cv2.cvtColor(img2_sharp, cv2.COLOR_BGR2RGB),
               img2_gray, img2_eq, img2_thresh, img2_erode, img2_dilate]

plt.figure(figsize=(15, 12))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    cmap_val = "gray" if len(images_list[i].shape) == 2 else None
    plt.imshow(images_list[i], cmap=cmap_val)
    plt.title(titles[i])
    plt.axis("off")
plt.tight_layout()
plt.savefig("outputs/task_comparison.png")
plt.show()

--- 
## Part J — Lab Questions

**1. What is image filtering?**  
*Answer:* Image filtering is a neighborhood operation where the value of any given pixel in the output image is computed by applying a mathematical function or matrix kernel across its corresponding spatial neighborhood in the input image.

**2. Why does a filter use neighboring pixels?**  
*Answer:* Natural visual features (surfaces, textures, edges, transitions) exhibit strong local spatial correlation. Neighborhood information provides contextual gradients, enabling operations such as smoothing fluctuations or detecting sharp directional transitions.

**3. What is a kernel?**  
*Answer:* A kernel (or mask/structuring matrix) is a small 2D matrix of predefined numerical coefficients used in spatial convolution and cross-correlation to weight neighboring pixels.

**4. What is convolution?**  
*Answer:* 2D discrete convolution involves sliding a flipped kernel over an input image, computing the element-wise product sum of overlapping pixels at each position to generate the filtered response.

**5. How does smoothing affect an image?**  
*Answer:* Smoothing attenuates high-frequency spatial components, reducing noise and local intensity variance at the cost of blurring sharp boundaries and losing fine textures.

**6. What is the difference between mean and Gaussian filtering?**  
*Answer:* Mean filtering assigns uniform weights ($1/N$) to every pixel in the window, often creating boxy artifacts. Gaussian filtering assigns weights inversely proportional to Euclidean distance from the center via a Gaussian distribution, yielding smoother and more isotropic results.

**7. Why can smoothing remove image details?**  
*Answer:* Fine details, thin lines, and sharp edges are high-frequency components. Averaging adjacent pixels blends local gradients, flattening the sharp transitions necessary to resolve detail.

**8. What is the purpose of sharpening?**  
*Answer:* Sharpening amplifies high-frequency differences (edges and contours) by subtracting local second derivatives (e.g., Laplacian) or unsharp masking, boosting subjective clarity.

**9. Why can sharpening increase the visibility of noise?**  
*Answer:* High-frequency spatial noise resembles localized edge transitions. Sharpening kernels accentuate rapid local variations indiscriminately, boosting both authentic edges and sensor noise.

**10. What information does an image histogram provide?**  
*Answer:* It plots the global frequency distribution of pixel intensity values, revealing exposure levels, contrast width, and dynamic range occupancy.

**11. What is histogram equalization?**  
*Answer:* A transformation technique that flattens and stretches the cumulative distribution function (CDF) of pixel intensities across the full available dynamic range (0–255), boosting global contrast.

**12. What is thresholding?**  
*Answer:* A point-wise segmentation technique mapping continuous grayscale intensities to a discrete binary representation based on whether they fall above or below a threshold $T$.

**13. What is the purpose of a binary image?**  
*Answer:* It isolates regions of interest (foreground objects vs. background), simplifying downstream tasks like contour extraction, blob detection, and morphological shape analysis.

**14. What is a structuring element?**  
*Answer:* A defined matrix shape (box, cross, ellipse) with an origin that specifies which neighboring pixels are evaluated during morphological operations.

**15. What is the difference between erosion and dilation?**  
*Answer:* Erosion outputs a foreground pixel only if the structuring element completely fits within the foreground (shrinking boundaries). Dilation outputs a foreground pixel if at least one element pixel overlaps the foreground (expanding boundaries).

**16. What is the difference between morphological opening and closing?**  
*Answer:* Opening is erosion followed by dilation, eliminating small foreground noise/islands. Closing is dilation followed by erosion, bridging gaps, cracks, and filling small holes within foreground bodies.

--- 
## Part K — Challenge Task

**Pipeline Architecture:**
$$\text{Original} \longrightarrow \text{Grayscale} \longrightarrow \text{Gaussian Filtering} \longrightarrow \text{Histogram Enhancement} \longrightarrow \text{Thresholding} \longrightarrow \text{Morphological Processing} \longrightarrow \text{Final Image}$$

In [ ]:
# Step 1: Input
step1_orig = cv2.imread("images/sample.jpg")

# Step 2: Grayscale conversion (Reduces dimensionality to single channel intensity)
step2_gray = cv2.cvtColor(step1_orig, cv2.COLOR_BGR2GRAY)

# Step 3: Gaussian Filtering (Suppresses high-frequency sensor noise before enhancement)
step3_gauss = cv2.GaussianBlur(step2_gray, (5, 5), 1.0)

# Step 4: Histogram Enhancement (Stretches dynamic range for better threshold separation)
step4_equalized = cv2.equalizeHist(step3_gauss)

# Step 5: Otsu's Thresholding (Separates foreground from background adaptively)
_, step5_thresh = cv2.threshold(step4_equalized, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

# Step 6: Morphological Processing (Opening followed by Closing to remove speckles and fill voids)
morph_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
step6_morph = cv2.morphologyEx(step5_thresh, cv2.MORPH_OPEN, morph_kernel)
step7_final = cv2.morphologyEx(step6_morph, cv2.MORPH_CLOSE, morph_kernel)

# Visualization of Pipeline
pipeline_steps = [
    ("1. Original", cv2.cvtColor(step1_orig, cv2.COLOR_BGR2RGB), None),
    ("2. Grayscale", step2_gray, "gray"),
    ("3. Gaussian Blur", step3_gauss, "gray"),
    ("4. Hist Equalized", step4_equalized, "gray"),
    ("5. Thresholding", step5_thresh, "gray"),
    ("6. Final Cleaned (Morph)", step7_final, "gray")
]

plt.figure(figsize=(18, 10))
for idx, (title, img_data, cmap) in enumerate(pipeline_steps):
    plt.subplot(2, 3, idx + 1)
    plt.imshow(img_data, cmap=cmap)
    plt.title(title, fontsize=12, fontweight="bold")
    plt.axis("off")

plt.tight_layout()
plt.savefig("outputs/challenge_pipeline.png")
plt.show()

### Pipeline Step Justifications
1. **Grayscale Conversion:** Strips redundant chromatic data to focus on scalar intensity values, lowering compute complexity.
2. **Gaussian Filtering:** Removes high-frequency noise spikes before contrast stretching so noise does not get amplified.
3. **Histogram Equalization:** Maximizes separation between foreground and background intensity peaks.
4. **Thresholding (Otsu):** Converts scalar intensity gradients into clean binary masks based on minimum intraclass variance.
5. **Morphological Filtering:** Eliminates spurious isolated background noise points and closes internal holes in target masks.

--- 
## Part L & M — Expected Outcomes and Submission Checklist

### Submission Requirements Checklist
- [x] Python Notebook (`.ipynb`) with complete executable workflow
- [x] Original sample image(s) configured in `images/` directory
- [x] Processed outputs and comparison matrices rendered
- [x] Completed comparison table
- [x] Complete technical answers to lab questions (Part J)
- [x] Independent practical task execution and verification (Part I)
- [x] Challenge task pipeline implementation and justification (Part K)